# CrickIQ — Exploratory Data Analysis

This notebook explores the synthetic IPL dataset generated for CrickIQ.

**Contents**
1. Data loading and overview
2. Match-level EDA
3. Team performance analysis
4. Toss impact analysis
5. Player performance EDA
6. Venue analysis
7. Season trends
8. Feature engineering preview

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

plt.style.use('dark_background')
sns.set_theme(style='darkgrid')
%matplotlib inline

print('Libraries loaded ✓')

## 1. Data Loading

In [ ]:
from utils.preprocessor import load_matches, load_deliveries

matches = load_matches()
deliveries = load_deliveries()

print(f'Matches   : {len(matches):,} rows × {matches.shape[1]} cols')
print(f'Deliveries: {len(deliveries):,} rows × {deliveries.shape[1]} cols')
matches.head(3)

In [ ]:
matches.describe()

In [ ]:
matches.info()

## 2. Match-Level EDA

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Matches per season
per_season = matches.groupby('season').size()
axes[0].bar(per_season.index, per_season.values, color='#2ECC71')
axes[0].set_title('Matches Per Season', color='white')
axes[0].set_xlabel('Season')
axes[0].set_ylabel('Matches')

# Result distribution
result_counts = matches['result'].value_counts()
axes[1].pie(result_counts, labels=result_counts.index, autopct='%1.1f%%',
            colors=['#2ECC71', '#3498DB', '#E74C3C'])
axes[1].set_title('Match Result Types', color='white')

plt.tight_layout()
plt.show()

## 3. Team Performance

In [ ]:
wins = matches[matches['winner'] != ''].groupby('winner').size().sort_values(ascending=False)
print('All-time wins by team:')
print(wins.to_string())

fig = px.bar(wins.reset_index(), x='winner', y=0,
             labels={'winner': 'Team', '0': 'Wins'},
             title='All-Time IPL Wins',
             color_discrete_sequence=['#2ECC71'],
             template='plotly_dark')
fig.show()

## 4. Toss Impact Analysis

In [ ]:
normal = matches[matches['result'] == 'normal'].copy()
normal['toss_match_win'] = (normal['toss_winner'] == normal['winner']).astype(int)

toss_impact = normal.groupby('toss_decision')['toss_match_win'].mean() * 100
print('Win % when batting first vs fielding first:')
print(toss_impact.round(1))

overall_toss_win = normal['toss_match_win'].mean() * 100
print(f'\nOverall toss winner match win %: {overall_toss_win:.1f}%')

## 5. Player Performance EDA

In [ ]:
from utils.preprocessor import compute_batting_stats, compute_bowling_stats

batting = compute_batting_stats(deliveries)
bowling = compute_bowling_stats(deliveries)

print('Top 10 Batsmen by Runs:')
print(batting[['batsman', 'total_runs', 'average', 'strike_rate']].head(10).to_string(index=False))

print('\nTop 10 Bowlers by Wickets:')
print(bowling[['bowler', 'wickets', 'economy', 'bowling_average']].head(10).to_string(index=False))

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

top_bat = batting.head(10)
ax1.barh(top_bat['batsman'], top_bat['total_runs'], color='#2ECC71')
ax1.set_title('Top 10 Batsmen — Runs', color='white')
ax1.invert_yaxis()

top_bowl = bowling.head(10)
ax2.barh(top_bowl['bowler'], top_bowl['wickets'], color='#E74C3C')
ax2.set_title('Top 10 Bowlers — Wickets', color='white')
ax2.invert_yaxis()

plt.tight_layout()
plt.show()

## 6. Venue Analysis

In [ ]:
venue_counts = matches['venue'].value_counts().head(10)
print('Top 10 venues by matches hosted:')
print(venue_counts.to_string())

## 7. Season Trends

In [ ]:
innings_scores = (
    deliveries.groupby(['match_id', 'inning'])['total_runs']
    .sum()
    .reset_index()
    .rename(columns={'total_runs': 'score'})
)
innings_scores = innings_scores.merge(matches[['id', 'season']], left_on='match_id', right_on='id')
season_avg = innings_scores.groupby('season')['score'].mean().reset_index()

fig = px.line(season_avg, x='season', y='score',
              title='Average Innings Score Per Season',
              markers=True,
              template='plotly_dark',
              color_discrete_sequence=['#2ECC71'])
fig.show()

## 8. Feature Engineering Preview

In [ ]:
from utils.feature_engineering import build_features, FEATURE_COLUMNS

print('Building features (may take ~30 s) …')
feat_df = build_features(matches)
print(f'Feature matrix shape: {feat_df[FEATURE_COLUMNS].shape}')
feat_df[FEATURE_COLUMNS].describe().round(3)

In [ ]:
import seaborn as sns

corr = feat_df[FEATURE_COLUMNS + ['team1_won']].corr()
plt.figure(figsize=(12, 8))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdYlGn', center=0,
            linewidths=0.5)
plt.title('Feature Correlation Matrix', color='white')
plt.tight_layout()
plt.show()

---
**CrickIQ EDA complete.** Run `streamlit run app.py` from the project root to launch the full dashboard.